# dim_comment_scd1 (SCD 1)

One row per YouTube comment. `like_count` / `reply_count` = latest read (type 1). `video_key` / `commenter_key` from `dim_video_scd1` / `dim_commenter_scd1` (−1 when missing), so this notebook runs after those two.

Task `dim_comment_scd1` of the job "2. gold (poc tables)"; widgets `load_type` (incremental / full), `from_date`, `to_date` (Vietnam days, empty = yesterday .. tomorrow). The live dim is unchanged.

In [ ]:
%run ./_scd

# Set up environment

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow

silver = dbutils.widgets.get("silver")
gold = dbutils.widgets.get("gold")
load_type = dbutils.widgets.get("load_type")
from_date, to_date = get_incremental_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))

# First load of the table = full (poc update_load_type: full once, then incremental)
if not spark.catalog.tableExists(f"{gold}.dim_comment_scd1"):
    load_type = "full"
print(f"load_type = {load_type}, window = [{from_date}, {to_date})")

# Read Source Table

In [ ]:
path = f"{gold}.dim_comment_scd1"
stg_comments = spark.table(f"{silver}.stg_comments")
stg_comments_latest = spark.table(f"{silver}.stg_comments_latest").filter("comment_id IS NOT NULL")
dim_video = spark.table(f"{gold}.dim_video_scd1").filter("video_key <> -1")
dim_commenter = spark.table(f"{gold}.dim_commenter_scd1").filter("commenter_key <> -1")

## Incremental: comments read in the window **+** comments still pointing to −1

A comment can arrive before its video / author reaches the dim (late-arriving dimension); those rows are re-checked every run.

In [ ]:
if load_type == "incremental":
    changed = (stg_comments
               .filter((F.col("snapshot_date") >= from_date) & (F.col("snapshot_date") < to_date))
               .select("comment_id"))
    if spark.catalog.tableExists(path):
        unresolved = (spark.table(path)
                      .filter("comment_key <> -1 AND (video_key = -1 OR commenter_key = -1)")
                      .select("comment_id"))
        changed = changed.union(unresolved)
    stg_comments_latest = stg_comments_latest.join(changed.distinct(), "comment_id", "left_semi")

# Transformation

In [ ]:
commenter_nk = F.coalesce(F.col("comment.author_channel_id"), F.concat(F.lit("name:"), F.col("comment.author_display_name")))

result_df = (
    stg_comments_latest.alias("comment")
    .join(dim_video.alias("video"), on=F.col("comment.video_id") == F.col("video.video_id"), how="left")
    .join(dim_commenter.alias("author"), on=commenter_nk == F.col("author.commenter_nk"), how="left")
    .select(
        F.xxhash64("comment.comment_id").alias("comment_key"),
        F.col("comment.comment_id").cast("string").alias("comment_id"),
        F.coalesce(F.col("video.video_key"), F.lit(-1)).cast("bigint").alias("video_key"),
        F.col("comment.video_id").cast("string").alias("video_id"),
        F.coalesce(F.col("author.commenter_key"), F.lit(-1)).cast("bigint").alias("commenter_key"),
        F.col("comment.author_display_name").cast("string").alias("author_display_name"),
        F.col("comment.published_at").cast("timestamp").alias("posted_at"),
        F.date_format(F.from_utc_timestamp("comment.published_at", TZ), "yyyyMMdd").cast("int").alias("posted_date_key"),
        F.col("comment.text").cast("string").alias("comment_text"),
        F.col("comment.like_count").cast("bigint").alias("like_count"),
        F.col("comment.total_reply_count").cast("int").alias("reply_count"),
        F.col("comment.created_at").cast("timestamp").alias("created_at"),
        F.col("comment.snapshot_ts").cast("timestamp").alias("modified_at"),
    )
)

## Add metadata

In [ ]:
insert_date = datetime.now()
etl_by = "Databricks"
result_df = result_df.withColumn("etl_date", F.lit(insert_date)).withColumn("etl_by", F.lit(etl_by))

## Add checksums

In [ ]:
primary_key = ["comment_id"]
result_df = add_checksums(result_df, primary_key, exclude=["comment_key", "created_at", "modified_at"])

## Create table

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {path} (
        comment_key BIGINT, comment_id STRING, video_key BIGINT, video_id STRING, commenter_key BIGINT,
        author_display_name STRING, posted_at TIMESTAMP, posted_date_key INT, comment_text STRING, like_count BIGINT,
        reply_count INT, created_at TIMESTAMP, modified_at TIMESTAMP, etl_date TIMESTAMP, etl_by STRING,
        primary_checksum STRING, record_checksum STRING
    ) USING DELTA
    COMMENT 'Comment dimension, SCD 1 (poc checksum method). like_count / reply_count = latest read. comment_key = xxhash64(comment_id), -1 = Unknown.'
""")
result_df = result_df.select(*spark.table(path).columns)

# Write to Table

**full**: overwrite (poc `full_refresh`). **incremental**: MERGE on `primary_checksum`; update only when `record_checksum` changed and the source row is not older (`s.modified_at >= t.modified_at`, so a backfill of an old day never overwrites newer values); insert new keys.

In [ ]:
if load_type == "full":
    full_refresh(result_df, path)
else:
    result_df.createOrReplaceTempView("source")
    spark.sql(f"""
        MERGE INTO {path} AS t
        USING source AS s
        ON t.primary_checksum = s.primary_checksum
        WHEN MATCHED AND t.record_checksum <> s.record_checksum
             AND (t.modified_at IS NULL OR s.modified_at >= t.modified_at) THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

## Unknown row

Key −1, natural key NULL: facts with no match point to it. Inserted only when missing (no empty commit).

In [ ]:
if spark.table(path).where("comment_key = -1").isEmpty():
    spark.sql(f"INSERT INTO {path} (comment_key, comment_text, video_key, commenter_key, etl_date, etl_by) VALUES (-1, 'Unknown', -1, -1, current_timestamp(), 'Databricks')")

## Check

In [ ]:
dup = spark.sql(f"SELECT COUNT(*) - COUNT(DISTINCT comment_id) FROM {path} WHERE comment_id IS NOT NULL").first()[0]
assert dup == 0, f"{path}: {dup} duplicate comment_id"
last = spark.sql(f"DESCRIBE HISTORY {path} LIMIT 2").where("operation <> 'WRITE'").select("operation", "operationMetrics").first()
print(f"✅ {path}: {spark.table(path).count()} rows", "·", last["operation"], dict(last["operationMetrics"] or {}) if last else "")